In [1]:
import os
import openprotein
from openprotein.protein import Protein
from openprotein.chains import DNA
from dotenv import load_dotenv
from CIF_to_PDB import convert_cif_to_pdb
from molviewspec import create_builder

load_dotenv()

def fold(session, name, p_seq, dna_seq):
    """Run one fold job for the provided protein sequence and DNA sequence(s).

    p_seq: protein sequence (str or bytes)
    dna_seq: either a single DNA sequence (str) or a list of DNA sequence strings
    """
    proteins = [Protein(sequence=p_seq)]
    proteins[0].chain_id = ["A", "B"]

    # Accept either a single DNA sequence or a list of sequences
    if isinstance(dna_seq, (str, bytes)):
        dna_list = [dna_seq]
    else:
        dna_list = list(dna_seq)

    chain_ids = ["C", "D", "E", "F", "G", "H", "I", "J", "K", "L", "M", "N",
                 "O", "P", "Q", "R", "S", "T", "U", "V", "W", "X", "Y", "Z"]
    dnas = []
    for dna in dna_list:
        dnas.append(DNA(sequence=dna, chain_id=chain_ids.pop()))

    # Build MSA seed: handle bytes or str sequences
    msa_query = []
    for p in proteins:
        seq = getattr(p, "sequence", "")
        if isinstance(seq, (bytes, bytearray)):
            seq_str = seq.decode()
        else:
            seq_str = str(seq)

        if p.chain_id is not None and isinstance(p.chain_id, list):
            for _ in p.chain_id:
                msa_query.append(seq_str)
        else:
            msa_query.append(seq_str)

    msa = session.align.create_msa(seed=":".join(msa_query))

    for p in proteins:
        p.msa = msa

    fold_job = session.fold.boltz2.fold(
        proteins=proteins,
        dnas=dnas
    )

    fold_job.wait_until_done(verbose=True)

    result = fold_job.get()

    builder = create_builder()
    structure = builder.download(url="mystructure.cif")\
        .parse(format="mmcif")\
        .model_structure()\
        .component()\
        .representation()\
        .color(color="blue")
    builder.molstar_notebook(data={'mystructure.cif': result}, width=500, height=400)

def main():
    """Connects to OpenProtein and runs all folding jobs."""
    user = os.environ.get("OPENPROTEIN_USERNAME")
    pw = os.environ.get("OPENPROTEIN_PASSWORD")

    session = openprotein.connect(username=user, password=pw, timeout=600)

    # protein = input("Enter the path to your protein file: ").strip()
    protein = "sequences/proteins/Fre1"

    names = []
    protein_list = []
    # Assume the data is in the form:
    #   - each protein to fold in its own file and are folded as a dimer
    #   - each file is one line of a protein sequence
    names.append(protein) # name of file will be the name of the fold
    with open(protein, 'r') as file:
        lines = file.readlines()
        protein_list.append(lines[0].strip()) # Add protein with list of proteins to fold and get rid of white space
    
    # Clean the DNA first:
    for dna_file in os.listdir("sequences/raw_dna"):
        file_path = os.path.join("sequences", "raw_dna", dna_file)
        if dna_file in os.listdir("sequences/dnas"):
            continue # Already cleaned
        else:
            with open(file_path, 'r') as file:
                d = []
                for line in file.readlines():
                    line = line.upper()
                    clean = line.replace('U', 'T') # Replace all instances of U with T
                    d.append(clean.strip()) # Add dna with list of dna and get rid of white space
            os.makedirs("sequences/dnas", exist_ok=True)
            file_path_w = os.path.join("sequences", "dnas", dna_file)
            with open(file_path_w, "w") as f:
                for i in d:
                    f.write(i + "\n")

    dna_list = []
    # Assume the data is in the form:
    #   - each dna to fold in its own file and are folded as a dimer
    for dna in os.listdir("sequences/dnas"):
        file_path = os.path.join("sequences", "dnas", dna)
        with open(file_path, 'r') as file:
            d = []
            for line in file.readlines():
                d.append(line.strip()) # Add dna with list of dna to fold and get rid of white space
            dna_list.append(d)

    # Assume every protein is to be folded with each dna sequence
    for name, protein in zip(names, protein_list):
        for j in range(len(dna_list)):
            foldname = name + str(dna_list[j]) # To identify which DNA the protein is folded with
            fold(session, foldname, protein, dna_list[j])

if __name__ == "__main__":
    main()

Waiting: 100%|██████████| 100/100 [01:00<00:00,  1.65it/s, status=SUCCESS]


<IPython.core.display.Javascript object>

Waiting: 100%|██████████| 100/100 [01:01<00:00,  1.63it/s, status=SUCCESS]


<IPython.core.display.Javascript object>

Waiting: 100%|██████████| 100/100 [01:01<00:00,  1.64it/s, status=SUCCESS]


<IPython.core.display.Javascript object>